# Tarea 1 – Parte 1: Recolección de noticias

- **Fecha de descarga:** 16-10-2026
- **Ventana de publicación:** 06-10-2026 al 16-10-2026
- **Medio elegido:** B - El Comercio
- **URLs base:** https://elcomercio.pe/

## 1. Revisión de robots.txt
Consulta del `robots.txt` de El Comercio con un User-Agent académico para verificar qué rutas se pueden rastrear y respetar las reglas del sitio.

In [1]:
import requests

url = "https://elcomercio.pe/robots.txt"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; TareaPUCP-QLab/1.0; scraping academico)"}

respuesta = requests.get(url, headers=headers)

print(respuesta.status_code)
print(respuesta.text)

200
# DIRECTIVAS GENERALES
User-agent: *
Disallow: /catalogo/
Disallow: /buscar/
Disallow: /archivo/todas/19*
Disallow: /archivo/todas/200*

# BLOQUEO DE SCRAPERS ANTIGUOS Y MALICIOSOS
User-agent: UbiCrawler
User-agent: DOC
User-agent: Zao
User-agent: sitecheck.internetseer.com
User-agent: Zealbot
User-agent: MSIECrawler
User-agent: SiteSnagger
User-agent: WebStripper
User-agent: WebCopier
User-agent: Fetch
User-agent: Offline Explorer
User-agent: Teleport
User-agent: TeleportPro
User-agent: WebZIP
User-agent: linko
User-agent: HTTrack
User-agent: Microsoft.URL.Control
User-agent: Xenu
User-agent: larbin
User-agent: libwww
User-agent: ZyBORG
User-agent: Download Ninja
User-agent: wget
User-agent: grub-client
User-agent: k2spider
User-agent: NPBot
User-agent: WebReaper
Disallow: /

# SITEMAPS 
Sitemap: https://elcomercio.pe/sitemap/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/politica/?outputType=xml
Sitemap: https://elcomercio.pe/sitemap/peru/?outputType=xml
Sitemap: https://

## 2. Extracción de 350 noticias, implementando pausas entre peticiones y guardando respaldo crudo. 

**Estrategia:**

1. **RSS por categoría** (`/arc/outboundfeeds/rss/category/<seccion>/`): entrega titular, bajada (`description`) y fecha (`pubDate`) sin visitar la nota.
2. **HTML de archivo por sección** (`/archivo/<seccion>/`) + **sitemap web**: dan ~100 notas recientes por sección. Las que no vinieron en el RSS se visitan una por una para sacar `h1` (titular), `meta description` (bajada) y `datePublished` (fecha). También se visitan los ítems del RSS que llegaron sin `description`.

Secciones elegidas por su cercanía a las 4 categorías del trabajo: `politica`, `economia`, `deporte-total`, `tecnologia`.

Pausa aleatoria de 1-2 s entre peticiones, máximo 1 reintento y respaldo crudo de cada respuesta en `datos/raw/`.

In [2]:
import gzip
import random
import re
import time
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE = "https://elcomercio.pe"
MEDIO = "El Comercio"
SECCIONES = ["politica", "economia", "deporte-total", "tecnologia"]
CUPO = 100  

COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha"]
DATOS = Path("datos")
RAW = DATOS / "raw" / datetime.now().strftime("%Y-%m-%d_%H%M")
RAW.mkdir(parents=True, exist_ok=True)
LIMA = timezone(timedelta(hours=-5))

sesion = requests.Session()
sesion.headers.update(headers) 

In [3]:
def pausa():
    time.sleep(random.uniform(1, 2))


def descargar(url, nombre_raw, comprimir=False):
    for intento in range(2):
        try:
            r = sesion.get(url, timeout=20)
            pausa()
            if r.status_code == 200:
                break
            print(f"  {r.status_code} {url}")
            if r.status_code in (404, 410):
                return None
        except requests.RequestException as e:
            pausa()
            print(f"  ERROR {url}: {e}")
    else:
        return None
    if comprimir:
        (RAW / (nombre_raw + ".gz")).write_bytes(gzip.compress(r.content))
    else:
        (RAW / nombre_raw).write_bytes(r.content)
    return r.content


def limpiar(texto):
    return " ".join(texto.split()) if texto else ""


def normalizar_url(href):
    url = href.split("?")[0].split("#")[0]
    if url.startswith("/"):
        url = BASE + url
    return url if url.startswith(BASE + "/") and url.endswith("-noticia/") else None


def seccion_de(url):
    return url.replace(BASE, "").strip("/").split("/")[0]

### 2.1 RSS por categoría
Descargamos el feed RSS de cada sección, extraemos los metadatos de cada noticia y las guardamos en `noticias.csv` sin duplicados.

In [4]:
def extraer_rss(seccion):
    contenido = descargar(f"{BASE}/arc/outboundfeeds/rss/category/{seccion}/?outputType=xml",
                          f"rss_{seccion}.xml")
    if contenido is None:
        return []
    filas = []
    for item in BeautifulSoup(contenido, "xml").find_all("item"):
        url = normalizar_url(item.link.text) if item.link else None
        if not url:
            continue
        fecha = parsedate_to_datetime(item.pubDate.text).astimezone(LIMA).strftime("%Y-%m-%d")
        filas.append({
            "titular": limpiar(item.title.text),
            "bajada": limpiar(item.description.text) if item.description else "",
            "medio": MEDIO,
            "seccion_original": seccion_de(url),
            "url": url,
            "fecha": fecha,
        })
    return filas


noticias = {} 
for seccion in SECCIONES:
    filas = extraer_rss(seccion)
    for f in filas[:CUPO]:
        noticias.setdefault(f["url"], f)
    print(f"RSS {seccion}: {len(filas)} ítems")

print(f"Total tras RSS: {len(noticias)}")

RSS politica: 37 ítems
RSS economia: 97 ítems
RSS deporte-total: 65 ítems
RSS tecnologia: 22 ítems
Total tras RSS: 221


### 2.2 HTML: archivo de cada sección + página de cada nota
Complementamos el RSS con enlaces del archivo y el sitemap hasta completar el cupo por sección, y rellenamos las bajadas faltantes visitando cada nota.

In [6]:
def links_seccion(seccion):
    hrefs = []
    contenido = descargar(f"{BASE}/archivo/{seccion}/", f"archivo_{seccion}.html")
    if contenido:
        hrefs += [a["href"] for a in BeautifulSoup(contenido, "html.parser").find_all("a", href=True)]
    contenido = descargar(f"{BASE}/sitemap/web/{seccion}/?outputType=xml", f"sitemap_{seccion}.xml")
    if contenido:
        hrefs += [loc.text for loc in BeautifulSoup(contenido, "xml").find_all("loc")]
    links = []
    for href in hrefs:
        url = normalizar_url(href)
        if url and seccion_de(url) == seccion and url not in links:
            links.append(url)
    return links


def extraer_nota(url):
    nombre = url.rstrip("/").split("/")[-1][:120] + ".html"
    contenido = descargar(url, nombre, comprimir=True)
    if contenido is None:
        return None
    soup = BeautifulSoup(contenido, "html.parser")
    h1 = soup.find("h1")
    meta = soup.find("meta", attrs={"name": "description"})
    fecha = re.search(r'"datePublished":"(\d{4}-\d{2}-\d{2})', contenido.decode("utf-8", "ignore"))
    if not h1:
        return None
    return {
        "titular": limpiar(h1.get_text()),
        "bajada": limpiar(meta["content"]) if meta and meta.get("content") else "",
        "medio": MEDIO,
        "seccion_original": seccion_de(url),
        "url": url,
        "fecha": fecha.group(1) if fecha else "",
    }


for seccion in SECCIONES:
    ya = sum(1 for f in noticias.values() if f["seccion_original"] == seccion)
    pendientes = [u for u in links_seccion(seccion) if u not in noticias][: max(0, CUPO - ya)]
    print(f"Archivo/sitemap {seccion}: {ya} desde RSS, {len(pendientes)} notas por visitar")
    for url in pendientes:
        fila = extraer_nota(url)
        if fila:
            noticias[url] = fila

sin_bajada = [u for u, f in noticias.items() if not f["bajada"]]
print(f"RSS sin bajada: {len(sin_bajada)} notas por visitar")
for url in sin_bajada:
    fila = extraer_nota(url)
    if fila and fila["bajada"]:
        noticias[url]["bajada"] = fila["bajada"]

print(f"Total tras HTML: {len(noticias)}")

Archivo/sitemap politica: 34 desde RSS, 58 notas por visitar
Archivo/sitemap economia: 96 desde RSS, 4 notas por visitar
Archivo/sitemap deporte-total: 64 desde RSS, 36 notas por visitar
Archivo/sitemap tecnologia: 21 desde RSS, 72 notas por visitar
RSS sin bajada: 23 notas por visitar
Total tras HTML: 391


### 2.3 Consolidado y guardado
Convertimos las noticias en un DataFrame, eliminamos vacíos y duplicados y guardamos la información en `noticias.csv`.

In [7]:
df = pd.DataFrame(list(noticias.values()), columns=COLUMNAS)
df = df[df["titular"].str.len() > 0].drop_duplicates(subset="url").reset_index(drop=True)
df.to_csv(DATOS / "noticias.csv", index=False, encoding="utf-8")

print(f"Noticias extraídas: {len(df)}  (mínimo requerido: 350)")
print(f"Respaldo crudo: {RAW}  ({len(list(RAW.iterdir()))} archivos)")
print(f"Sin bajada: {(df['bajada'] == '').sum()} | Sin fecha: {(df['fecha'] == '').sum()}")
print(f"Fechas de publicación: {df['fecha'].min()} a {df['fecha'].max()}")
print(df["seccion_original"].value_counts())

Noticias extraídas: 391  (mínimo requerido: 350)
Respaldo crudo: datos\raw\2026-10-16_1935  (205 archivos)
Sin bajada: 0 | Sin fecha: 0
Fechas de publicación: 2026-03-16 a 2026-10-16
seccion_original
economia          100
deporte-total     100
tecnologia         93
politica           92
ecdata              3
opinion             2
ruedas-tuercas      1
Name: count, dtype: int64


In [8]:
df.head(3)

,titular,bajada,medio,seccion_original,url,fecha
0,Viceministra de Minas: “Estamos estableciendo ...,"La viceministra de Minas, Mayra Figueroa, defi...",El Comercio,politica,https://elcomercio.pe/politica/gobierno/entrev...,2026-10-16
1,Ministro Marco Vinelli sobre facultades: “Se e...,"El ministro de Desarrollo Agrario y Riego, Mar...",El Comercio,politica,https://elcomercio.pe/politica/congreso/minist...,2026-10-16
2,Keiko Fujimori pide unidad ante fenómeno El Ni...,"La presidenta de la República, Keiko Fujimori,...",El Comercio,politica,https://elcomercio.pe/politica/gobierno/keiko-...,2026-10-16


## 3. Limpieza y normalización

### 3.1 Duplicados por URL y titulares vacíos
No se han cargado duplicados desde el inicio del código. Al definir noticias {} en el punto 2.1 como noticias.setdefault(f["url"], f), solo se agrega una línea si la URL no existe. Si fuesen duplicadas, se mantiene la primera y se ignora la última. Asimismo "if u not in noticias" del punto 2.2 no cuenta URL ya capturadas antes de visitar cada página. Finalmente en 2.3. el código df = df[df["titular"].str.len() > 0].drop_duplicates(subset="url").reset_index(drop=True) elimina duplicados sobre la base del url y los vacíos, manteniendo aquellas noticias con titular. 

### 3.2 Recategorización de `seccion_original` 
Las secciones que no corresponden a una categoría se descartan:

- `ecdata`: periodismo de datos que mezcla temas (elecciones, presupuesto, UFC).
- `opinion`: columnas de opinión, no noticias.
- `ruedas-tuercas`: automotriz.

In [9]:
# En el caso de tecnología incluye subsecciones ciencias, ecologia, inteligencia-artificial, etc.

MAPEO = {
    "politica": "politica",
    "economia": "economia",
    "deporte-total": "deportes",
    "tecnologia": "ciencia_tecnologia",  
}

equivalencias = (df["seccion_original"].value_counts()
                 .rename_axis("seccion_original").reset_index(name="noticias"))
equivalencias["etiqueta"] = equivalencias["seccion_original"].map(MAPEO).fillna("(descartada)")
equivalencias

,seccion_original,noticias,etiqueta
0,economia,100,economia
1,deporte-total,100,deportes
2,tecnologia,93,ciencia_tecnologia
3,politica,92,politica
4,ecdata,3,(descartada)
5,opinion,2,(descartada)
6,ruedas-tuercas,1,(descartada)


In [10]:
n_extraidas = len(noticias)
descartes = [("Vacío o URL duplicada", n_extraidas - len(df))]

df["etiqueta"] = df["seccion_original"].map(MAPEO)

fuera = df["etiqueta"].isna()
for seccion, n in df.loc[fuera, "seccion_original"].value_counts().items():
    descartes.append((f"Otra categoría ({seccion})", int(n)))
df = df[~fuera].reset_index(drop=True)

In [11]:
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["mes"] = df["fecha"].dt.to_period("M")

conteo = (df.groupby(["mes", "etiqueta"])
            .size()
            .unstack(fill_value=0))

conteo

etiqueta,ciencia_tecnologia,deportes,economia,politica
mes,,,,
2026-03,1,0,0,0
2026-04,0,0,0,1
2026-10,92,100,100,91


Nos quedamos solo las noticias publicadas en el mes de octubre (06-10-2026 al 16-10-2026)

In [12]:
mes = pd.to_datetime(df["fecha"]).dt.month
antiguas = mes.isin([3, 4])
descartes.append(("Publicada en marzo o abril", int(antiguas.sum())))
df = df[~antiguas].reset_index(drop=True)

### 3.3 Reporte de descartes y guardado

In [13]:
reporte = pd.DataFrame(descartes, columns=["motivo", "noticias"])
reporte = pd.concat([reporte, pd.DataFrame([{"motivo": "TOTAL descartadas", "noticias": reporte["noticias"].sum()}])],
                    ignore_index=True)

print(f"Extraídas: {n_extraidas} | Descartadas: {n_extraidas - len(df)} | Finales: {len(df)}  (mínimo: 350)")
reporte

Extraídas: 391 | Descartadas: 8 | Finales: 383  (mínimo: 350)


,motivo,noticias
0,Vacío o URL duplicada,0
1,Otra categoría (ecdata),3
2,Otra categoría (opinion),2
3,Otra categoría (ruedas-tuercas),1
4,Publicada en marzo o abril,2
5,TOTAL descartadas,8


In [14]:
COLUMNAS_FINALES = COLUMNAS + ["etiqueta"]
df = df[COLUMNAS_FINALES]
df.to_csv(DATOS / "noticias.csv", index=False, encoding="utf-8")

print(f"Guardado {DATOS / 'noticias.csv'}: {df.shape[0]} filas x {df.shape[1]} columnas")
print(df["etiqueta"].value_counts())

Guardado datos\noticias.csv: 383 filas x 7 columnas
etiqueta
economia              100
deportes              100
ciencia_tecnologia     92
politica               91
Name: count, dtype: int64


## 4. Análisis Exploratorio (EDA)

4.1. Distribución de noticias por categoría

In [23]:
import plotly.express as px

dist = (df["etiqueta"].value_counts()
          .rename_axis("categoria")
          .reset_index(name="noticias"))

fig = px.bar(dist, x="categoria", y="noticias",
             color="categoria", text="noticias",
             color_discrete_sequence=px.colors.sequential.Reds[-1:-8:-2],
             title=(f"<b>Distribución practicamente equitativa entre las categorías"),
             subtitle=(f"El Comercio - 06 Oct 2026 al 16 Oct 2026")
             )

fig.update_traces(textposition="outside")
fig.update_layout(title_x=0.5,
                  xaxis_title="<b>Categoría</b>",
                  yaxis_title="<b>Número de noticias</b>",
                  yaxis_range=[0, 120],
                  plot_bgcolor="white",
                  showlegend=False)
fig.update_yaxes(gridcolor="#E5E5E5")

fig.add_annotation(text="Fuente: El Comercio (elcomercio.pe), RSS y archivo web, descarga del 16/10/2026. Elaboración propia.",
                   xref="paper", yref="paper", x=0, y=-0.22,
                   xanchor="left", yanchor="top", showarrow=False,
                   font=dict(size=10, color="#555555"))
fig.update_layout(margin=dict(b=100))

fig.show()

4.2. Distribución de noticias por rango de fechas

In [22]:
diario = (df.groupby([df["fecha"].dt.date, "etiqueta"])
            .size()
            .reset_index(name="noticias"))
diario.columns = ["fecha", "categoria", "noticias"]
diario["fecha"] = pd.to_datetime(diario["fecha"]).dt.strftime("%d/%m")

rojos = px.colors.sequential.Reds[-1:-8:-2]
orden_cat = list(dist["categoria"])
colores = dict(zip(orden_cat, rojos))

orden_fechas = sorted(diario["fecha"].unique(), key=lambda d: (d[3:], d[:2]))

por_dia = diario.groupby("fecha")["noticias"].sum()
pct_3 = por_dia[orden_fechas[-3:]].sum() / por_dia.sum()
fechas = df["fecha"]

fig = px.bar(diario, x="fecha", y="noticias", color="categoria",
             color_discrete_map=colores,
             category_orders={"categoria": orden_cat, "fecha": orden_fechas},
             title=f"<b>El {pct_3:.0%} de las noticias se concentra en los últimos 3 días</b>",
             subtitle=(f"El Comercio - 06 Oct 2026 al 16 Oct 2026"))

fig.update_layout(title_x=0.5,
                  xaxis_title="<b>Fecha de publicación</b>",
                  yaxis_title="<b>Número de noticias</b>",
                  yaxis_range=[0, por_dia.max() * 1.15],
                  barmode="stack",
                  plot_bgcolor="white",
                  legend=dict(title=dict(text="<b>Categoría</b>"),
                              bgcolor="#F2F2F2", bordercolor="#D9D9D9", borderwidth=1),
                  margin=dict(b=100))
fig.update_xaxes(type="category")
fig.update_yaxes(gridcolor="#E5E5E5")

totales = por_dia[orden_fechas]
fig.add_scatter(x=orden_fechas, y=totales, text=totales, mode="text",
                textposition="top center", textfont=dict(size=11, color="#333333"),
                showlegend=False, hoverinfo="skip")

fig.add_annotation(text="Fuente: El Comercio, RSS y archivo web, descarga del 16/10/2026. Elaboración propia.",
                   xref="paper", yref="paper", x=0, y=-0.25,
                   xanchor="left", yanchor="top", showarrow=False,
                   font=dict(size=10, color="#555555"))

fig.show()

4.3 Histograma de longitud de caracteres de los titulares

In [34]:
largo = df["titular"].str.len().rename("caracteres").to_frame()
desde, hasta = 70, 120
pct = largo["caracteres"].between(desde, hasta).mean()
fechas = pd.to_datetime(df["fecha"])

fig = px.histogram(largo, x="caracteres", nbins=30, text_auto=True,
                   color_discrete_sequence=[px.colors.sequential.Reds[-2]],
                   title=f"<b>El {pct:.0%} de los titulares tiene entre {desde} y {hasta} caracteres</b>",
                   subtitle=f"<i>El Comercio - {fechas.min():%d %b %Y} al {fechas.max():%d %b %Y}</i>")

fig.update_traces(marker_line_color="white", marker_line_width=1,
                  textposition="outside")

fig.update_layout(title_x=0.5,
                  xaxis_title="<b>Número de caracteres del titular</b>",
                  yaxis_title="<b>Número de noticias</b>",
                  plot_bgcolor="white", bargap=0.05, margin=dict(b=100))
fig.update_yaxes(gridcolor="#E5E5E5", range=[0, 90])

fig.add_annotation(text="Fuente: El Comercio, RSS y archivo web. Elaboración propia.",
                   xref="paper", yref="paper", x=0, y=-0.25,
                   xanchor="left", yanchor="top", showarrow=False,
                   font=dict(size=10, color="#555555"))

fig.show()

## 5. Validación 

In [38]:
df["largo_titular"] = df["titular"].str.len()

cols = ["largo_titular", "etiqueta", "titular"]

print("Titulares MÁS LARGOS")
display(df.nlargest(5, "largo_titular")[cols])

Titulares MÁS LARGOS


,largo_titular,etiqueta,titular
307,173,deportes,Annia Cillóniz líder en su categoría en Camino...
167,171,deportes,“Con Mano Menezes el mejor jugador de la selec...
225,166,politica,Keiko Fujimori sobre Álvaro Vargas Llosa: “Con...
138,160,deportes,“El día de mañana me encantaría ver a tantas n...
221,160,politica,Juan Sheput sobre eventual suspensión del viaj...


Dos titulares tienen más de 170 caracteres y tres titulares están entre 160-169 caracteres, lo cual coincide con la representación gráfica del histograma. 

In [39]:
print("Titulares MÁS CORTOS")
display(df.nsmallest(5, "largo_titular")[cols])

Titulares MÁS CORTOS


,largo_titular,etiqueta,titular
77,22,economia,Invocación por el Perú
49,24,economia,IA y Sandbox regulatorio
78,41,economia,La evolución de la confianza en los pagos
80,46,economia,La gloria gastronómica que debemos multiplicar
332,59,ciencia_tecnologia,"Qué es la ""depresión sonriente"" y cómo puedes ..."


Dos titulares tienen entre 20-29 caracteres y dos titulares están entre 40-49 caracteres, lo cual coincide con la representación gráfica del histograma. 

## 6. Exportación

Se guarda el dataset en datos/noticias.csv

In [40]:
final = df[COLUMNAS_FINALES].copy()
final["fecha"] = pd.to_datetime(final["fecha"]).dt.strftime("%Y-%m-%d")

assert final.shape[1] == 7, f"Se esperaban 7 columnas, hay {final.shape[1]}"

final.to_csv(DATOS / "noticias.csv", index=False, encoding="utf-8")

print(f"Guardado {DATOS / 'noticias.csv'}")
print(f"df.shape: {final.shape}")
final.head(5)

Guardado datos\noticias.csv
df.shape: (383, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Viceministra de Minas: “Estamos estableciendo ...,"La viceministra de Minas, Mayra Figueroa, defi...",El Comercio,politica,https://elcomercio.pe/politica/gobierno/entrev...,2026-10-16,politica
1,Ministro Marco Vinelli sobre facultades: “Se e...,"El ministro de Desarrollo Agrario y Riego, Mar...",El Comercio,politica,https://elcomercio.pe/politica/congreso/minist...,2026-10-16,politica
2,Keiko Fujimori pide unidad ante fenómeno El Ni...,"La presidenta de la República, Keiko Fujimori,...",El Comercio,politica,https://elcomercio.pe/politica/gobierno/keiko-...,2026-10-16,politica
3,César Acuña anuncia su retiro de la política: ...,"“Les doy una buena noticia: chau, política”, a...",El Comercio,politica,https://elcomercio.pe/politica/partidos/cesar-...,2026-10-16,politica
4,Proyecto de Ley MAPE del gobierno propone que ...,A dos meses y medio para que concluya el Reinf...,El Comercio,politica,https://elcomercio.pe/politica/gobierno/el-pro...,2026-10-16,politica
